In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
train_data=pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test_data=pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

In [3]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [5]:
women=train_data.loc[train_data.Sex=='female']['Survived']
men=train_data.loc[train_data.Sex=='male']['Survived']
rate_women = sum(women) / len(women)
rate_men=sum(men)/len(men)
print(rate_women)
print(rate_men)

0.7420382165605095
0.18890814558058924


In [6]:
for pclass in [1,2,3]:
    assume=train_data.loc[(train_data.Sex=='female') & (train_data.Pclass==pclass)]['Survived'] 
    print(sum(assume)/len(assume))

0.9680851063829787
0.9210526315789473
0.5


In [7]:
train_data.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [8]:
work = train_data.copy()
work['Sex']=work['Sex'].map({'male':0,'female':1})
work[ "Embarked" ] = work[ "Embarked" ].fillna( "S" )
work[ "Embarked" ] = work[ "Embarked" ]. map ({ "S" : 0 , "C" : 1 , "Q" : 2 })
work[ "Age" ] = work[ "Age" ].fillna(work[ "Age" ].median())
print (work[[ "Sex" , "Embarked" , "Age" ]].head())

   Sex  Embarked   Age
0    0         0  22.0
1    1         1  38.0
2    1         0  26.0
3    1         0  35.0
4    0         0  35.0


In [9]:
features = [ "Pclass" , "Sex" , "Age" , "SibSp" , "Parch" , "Fare" , "Embarked" ]
X=work[features]
y=work['Survived']
print(X.head())
print(y.head())

   Pclass  Sex   Age  SibSp  Parch     Fare  Embarked
0       3    0  22.0      1      0   7.2500         0
1       1    1  38.0      1      0  71.2833         1
2       3    1  26.0      0      0   7.9250         0
3       1    1  35.0      1      0  53.1000         0
4       3    0  35.0      0      0   8.0500         0
0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=0)

print("number of people in the training set‌:", len(train_X))
print("number of people in the testing set‌:", len(val_X))

number of people in the training set‌: 668
number of people in the testing set‌: 223


In [11]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=1)
model.fit(train_X, train_y)

print("Model training is complete‌")

Model training is complete‌


In [12]:
from sklearn.metrics import accuracy_score

preds = model.predict(val_X)

print("test_score:", accuracy_score(val_y, preds))

test_score: 0.8251121076233184


In [13]:
importance = pd.Series(model.feature_importances_, index=features)
print(importance.sort_values(ascending=False))

Sex         0.268840
Fare        0.262609
Age         0.261674
Pclass      0.080258
SibSp       0.051098
Embarked    0.037959
Parch       0.037562
dtype: float64


In [14]:
for max_leaf_nodes in [5, 50, 500, 5000]:
    model = RandomForestClassifier(max_leaf_nodes=max_leaf_nodes, random_state=1)
    model.fit(train_X, train_y)
    preds = model.predict(val_X)
    score = accuracy_score(val_y, preds)
    print("Max leaf nodes: %d \t Accuracy: %g" % (max_leaf_nodes, score))

Max leaf nodes: 5 	 Accuracy: 0.820628
Max leaf nodes: 50 	 Accuracy: 0.852018
Max leaf nodes: 500 	 Accuracy: 0.829596
Max leaf nodes: 5000 	 Accuracy: 0.829596


In [15]:
work_test = test_data.copy()

work_test["Sex"] = work_test["Sex"].map({"male": 0, "female": 1})
work_test["Embarked"] = work_test["Embarked"].fillna("S")
work_test["Embarked"] = work_test["Embarked"].map({"S": 0, "C": 1, "Q": 2})
work_test["Age"] = work_test["Age"].fillna(work["Age"].median())
work_test["Fare"] = work_test["Fare"].fillna(work["Fare"].median())

X_test = work_test[features]
print(X_test.head())
print(X_test.shape)

   Pclass  Sex   Age  SibSp  Parch     Fare  Embarked
0       3    0  34.5      0      0   7.8292         2
1       3    1  47.0      1      0   7.0000         0
2       2    0  62.0      0      0   9.6875         2
3       3    0  27.0      0      0   8.6625         0
4       3    1  22.0      1      1  12.2875         0
(418, 7)


In [16]:
final_model = RandomForestClassifier(max_leaf_nodes=50, random_state=1)
final_model.fit(X, y)

test_preds = final_model.predict(X_test)

output = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": test_preds.astype(int),
})

output.to_csv("submission.csv", index=False)

print(output.head())
print("Total:", len(output))
print("Predicted survived:", test_preds.sum())

   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
Total: 418
Predicted survived: 134
